# 健康 vs 患者 MSN 边逐个检验（Edge-wise GLM）

本 notebook 对健康人与患者的 MSN 做**边水平**的线性模型检验：

- 数据与设计矩阵与 `health_patient_nbs.ipynb` 一致（协变量：group, sex, age_centered, global_centered）。
- 对每条边做 OLS，提取 group 系数的 t 与 p。
- 使用 **Benjamini–Hochberg FDR** 与 **Bonferroni** 做多重比较校正。
- 结果保存至 `results/edgewise_glm/health_patient_edgewise.csv`。

In [ ]:
# 环境与路径（与 health_patient_nbs 一致）
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "msn_pipeline").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

print("项目根:", ROOT)

In [ ]:
N_NODES = 148
N_EDGES = N_NODES * (N_NODES - 1) // 2  # 10878


def _n_edges(n: int) -> int:
    return n * (n - 1) // 2


def matrix_to_edges(A: np.ndarray) -> np.ndarray:
    """(n, n) -> (n_edges,) 或 (n_subj, n, n) -> (n_subj, n_edges)。上三角行优先。"""
    if A.ndim == 2:
        n = A.shape[0]
        return A[np.triu_indices(n, k=1)]
    if A.ndim == 3:
        n_subj, n, _ = A.shape
        E = _n_edges(n)
        out = np.empty((n_subj, E), dtype=A.dtype)
        for i in range(n_subj):
            out[i] = A[i][np.triu_indices(n, k=1)]
        return out
    raise ValueError("A must be 2D or 3D")


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats


N_NODES = 148
N_EDGES = N_NODES * (N_NODES - 1) // 2  # 10878
DATA_ROOT = ROOT / "data"
RESULTS_ROOT = ROOT / "results"
HEALTH_INFO_PATH = DATA_ROOT / "health_info_processed.csv"
PATIENT_INFO_PATH = DATA_ROOT / "patient_info_processed.csv"
HEALTH_MSN_DIR = RESULTS_ROOT / "health_msn_matrices"
PATIENT_MSN_DIR = RESULTS_ROOT / "patient_msn_matrices"
EDGEWISE_OUT_DIR = RESULTS_ROOT / "edgewise_glm"
EDGEWISE_OUT_DIR.mkdir(parents=True, exist_ok=True)
EDGEWISE_CSV = EDGEWISE_OUT_DIR / "health_patient_edgewise.csv"

## 1. 加载协变量与 MSN 矩阵

只保留在 info 表中有记录且存在对应 MSN 文件的被试；Gender：f→0, m→1。

In [ ]:
def load_msn_matrix(csv_file: Path, n_roi: int = 148) -> np.ndarray:
    """读取 MSN CSV，提取 148×148 数值矩阵（跳过首行首列标签）。"""
    df = pd.read_csv(csv_file, header=None)
    return df.iloc[1 : n_roi + 1, 1 : n_roi + 1].values.astype(np.float64)


# 健康人：info + MSN（{id}_feature_matrix_msn.csv）
health_info = pd.read_csv(HEALTH_INFO_PATH, dtype={"ID": str, "Age": np.float64})

health_ids = []
health_matrices = []
health_ages = []
health_sexes = []
for _, row in health_info.iterrows():
    sid = str(row["ID"]).strip()
    f = HEALTH_MSN_DIR / f"{sid}_feature_matrix_msn.csv"
    if not f.exists():
        continue
    health_ids.append(sid)
    health_matrices.append(load_msn_matrix(f, N_NODES))
    health_ages.append(row["Age"])
    health_sexes.append(row["Sex"])

if health_matrices:
    health_matrices = np.stack(health_matrices, axis=0)
    health_ages = np.asarray(health_ages, dtype=np.float64)
    health_sexes = np.asarray(health_sexes, dtype=np.float64)
else:
    health_matrices = np.empty((0, N_NODES, N_NODES), dtype=np.float64)
    health_ages = np.array([], dtype=np.float64)
    health_sexes = np.array([], dtype=np.float64)

n_health = len(health_ids)
print(f"健康人: {n_health} 人，矩阵形状 {health_matrices.shape}")

In [ ]:
# 患者：info + MSN（{id}_msn_matrix.csv）；ID 与文件名一致
patient_info = pd.read_csv(PATIENT_INFO_PATH, dtype={"ID": str, "Age": np.float64})

patient_ids = []
patient_matrices = []
patient_ages = []
patient_sexes = []
for _, row in patient_info.iterrows():
    sid = str(row["ID"]).strip()
    f = PATIENT_MSN_DIR / f"{sid}_msn_matrix.csv"
    if not f.exists():
        continue
    patient_ids.append(sid)
    patient_matrices.append(load_msn_matrix(f, N_NODES))
    patient_ages.append(row["Age"])
    patient_sexes.append(row["Sex"])

if patient_matrices:
    patient_matrices = np.stack(patient_matrices, axis=0)
    patient_ages = np.asarray(patient_ages, dtype=np.float64)
    patient_sexes = np.asarray(patient_sexes, dtype=np.float64)
else:
    patient_matrices = np.empty((0, N_NODES, N_NODES), dtype=np.float64)
    patient_ages = np.array([], dtype=np.float64)
    patient_sexes = np.array([], dtype=np.float64)

n_patient = len(patient_ids)
print(f"患者: {n_patient} 人，矩阵形状 {patient_matrices.shape}")

## 2. 构建设计矩阵

合并健康/患者，计算 age_centered、global_centered；设计矩阵列：[1, group, sex, age_centered, global_centered]，对比 [0,1,0,0,0] 为 group 主效应。

In [ ]:
# 合并：健康在前，患者在后
matrices_all = np.concatenate([health_matrices, patient_matrices], axis=0)
n_all = matrices_all.shape[0]
group = np.concatenate([np.zeros(n_health, dtype=np.float64), np.ones(n_patient, dtype=np.float64)])
sex_all = np.concatenate([health_sexes, patient_sexes])
age_all = np.concatenate([health_ages, patient_ages])

# 缺失 Age/Gender 用全样本均值插补
for arr in (sex_all, age_all):
    mask = np.isnan(arr)
    if mask.any():
        arr[mask] = np.nanmean(arr[~mask])

global_strength = matrices_all.reshape(n_all, -1).mean(axis=1).astype(np.float64)
global_centered = global_strength - global_strength.mean()
age_centered = age_all - age_all.mean()

design = np.column_stack([
    np.ones(n_all, dtype=np.float64),
    group,
    sex_all,
    age_centered,
    global_centered,
])
contrast = np.array([0.0, 1.0, 0.0, 0.0, 0.0], dtype=np.float64)

print("合并后被试数:", n_all)
print("Design shape:", design.shape)
print("Contrast (group):", contrast)

## 3. 边水平 OLS 与 t / p

将矩阵展平为上三角边向量，对每条边做 OLS，得到 group 系数的 t 与双侧 p。

In [ ]:
# 边矩阵 (n_all, n_edges)
edge_matrix = matrix_to_edges(matrices_all)
n_edges = edge_matrix.shape[1]
assert n_edges == N_EDGES

X = design.astype(np.float64)
Y = edge_matrix.astype(np.float64)
n, p = X.shape
df_resid = n - p

# OLS: B = (X'X)^{-1} X'Y
XtX = X.T @ X
iXtX = np.linalg.inv(XtX)
B = iXtX @ (X.T @ Y)
R = Y - X @ B
MSE = np.sum(R * R, axis=0) / np.maximum(df_resid, 1)

# 对比 c'B 的 t 统计量
c = contrast.ravel()
effect = c @ B
c_iXtX_c = float(c @ iXtX @ c)
se = np.sqrt(np.maximum(c_iXtX_c * MSE, np.finfo(float).tiny))
t_vec = effect / se

# 双侧 p 值
p_uncorrected = 2 * stats.t.sf(np.abs(t_vec), df_resid)

print("边数:", n_edges)
print("t: min = %.3f, max = %.3f" % (t_vec.min(), t_vec.max()))
print("p (uncorrected): min = %.2e, 中位数 = %.2e" % (p_uncorrected.min(), np.median(p_uncorrected)))

## 4. 多重比较校正（FDR + Bonferroni）

对 10878 个 p 值做 Benjamini–Hochberg FDR（alpha=0.05）与 Bonferroni 校正。

In [ ]:
from msn_stats.fdr import fdr_correct

_, p_fdr = fdr_correct(p_uncorrected, alpha=0.05)

# Bonferroni: p_bonf = min(p * n_edges, 1)
p_bonf = np.minimum(p_uncorrected * n_edges, 1.0)

alpha = 0.01
significant_fdr = p_fdr <= alpha
significant_bonf = p_bonf <= alpha

print("FDR (alpha=0.01) 显著边数:", significant_fdr.sum())
print("Bonferroni (alpha=0.01) 显著边数:", significant_bonf.sum())

## 5. 边索引与 (node_i, node_j)

将 edge_index 映射为 (i, j)，与 matrix_to_edges 的上三角顺序一致（i < j）。

In [ ]:
def edge_index_to_ij(edge_idx: int, n_nodes: int):
    """线性边索引 -> (i, j)，i < j，与 matrix_to_edges 约定一致。"""
    i = 0
    start = 0
    while i < n_nodes:
        count = n_nodes - 1 - i
        if edge_idx < start + count:
            j = i + 1 + (edge_idx - start)
            return (i, j)
        start += count
        i += 1
    raise IndexError("edge_idx out of range")

node_i = np.array([edge_index_to_ij(k, N_NODES)[0] for k in range(n_edges)])
node_j = np.array([edge_index_to_ij(k, N_NODES)[1] for k in range(n_edges)])

## 6. 结果表保存与展示

DataFrame 含 edge_index, node_i, node_j, t, p_uncorrected, p_fdr, p_bonf, significant_fdr, significant_bonf；保存为 CSV。

In [ ]:
results_df = pd.DataFrame({
    "edge_index": np.arange(n_edges),
    "node_i": node_i,
    "node_j": node_j,
    "t": t_vec,
    "p_uncorrected": p_uncorrected,
    "p_fdr": p_fdr,
    "p_bonf": p_bonf,
    "significant_fdr": significant_fdr,
    "significant_bonf": significant_bonf,
})

# results_df.to_csv(EDGEWISE_CSV, index=False)
print("已保存:", EDGEWISE_CSV)
print("\n前 10 行:")
display(results_df.head(10))
print("\nFDR 显著边数:", results_df["significant_fdr"].sum())
print("Bonferroni 显著边数:", results_df["significant_bonf"].sum())

## 7. Bonferroni 子集 + 效应量阈值（少而稳的边）

在 Bonferroni 显著的边中，再按 **效应量**（group 系数的绝对值 |β|）过滤，只保留效应量不低于阈值的边，得到一组“少而稳”的边用于主结论或可视化。效应量阈值可设为 Bonferroni 子集内 |effect| 的百分位数（如 75%），或直接指定数值。

In [ ]:
# 效应量 = group 系数（患者相对健康的边强度差），来自 OLS 的 contrast @ B
results_df["effect"] = effect

# 先取 Bonferroni 显著子集
bonf_subset = results_df[results_df["significant_bonf"]].copy()
n_bonf = len(bonf_subset)
if n_bonf == 0:
    print("Bonferroni 显著边数为 0，跳过效应量筛选。")
    stable_edges_df = pd.DataFrame()
else:
    # 效应量阈值：取 Bonferroni 子集内 |effect| 的 75% 分位数（保留效应量较大的 25% 边）
    # 可改为 50、90 等，或直接用固定值 EFFECT_SIZE_THRESHOLD = 0.05
    EFFECT_PERCENTILE = 99  # 只保留 |effect| >= 该百分位数的边
    effect_threshold = np.percentile(np.abs(bonf_subset["effect"]), EFFECT_PERCENTILE)
    stable_edges_df = bonf_subset[np.abs(bonf_subset["effect"]) >= effect_threshold].copy()
    stable_edges_df = stable_edges_df.sort_values("p_bonf").reset_index(drop=True)

    EDGEWISE_STABLE_CSV = EDGEWISE_OUT_DIR / "health_patient_edgewise_stable.csv"
    stable_edges_df.to_csv(EDGEWISE_STABLE_CSV, index=False)
    print("Bonferroni 显著边数:", n_bonf)
    print("效应量阈值 |effect| >= %.4f (对应 Bonferroni 子集内 %d%% 分位数)" % (effect_threshold, EFFECT_PERCENTILE))
    print("少而稳边数 (Bonferroni + 效应量阈值):", len(stable_edges_df))
    print("已保存:", EDGEWISE_STABLE_CSV)
    display(stable_edges_df.head(10))

## 8. 绘制 Bonferroni 显著边的群体 MSN（健康 / 患者）

使用 `stable_edges_df`（Bonferroni + 效应量阈值）中的边，分别绘制**健康人**与**患者**的群体 MSN：仅保留这些显著边上的连接强度（取各组均值），其余边置 0，参考 `绘制群体MSN.ipynb` 用 nilearn 的 `plot_connectome` 出图。

In [ ]:
# 依赖：nilearn（atlas 与 plot_connectome）
from nilearn.datasets import fetch_atlas_destrieux_2009
from nilearn.plotting import find_parcellation_cut_coords
from nilearn import plotting
import matplotlib.pyplot as plt

if len(stable_edges_df) == 0:
    print("stable_edges_df 为空，跳过绘图。请先运行上方「Bonferroni + 效应量阈值」单元。")
else:
    # 群体 MSN：健康组 / 患者组在每条边上的均值
    health_group_mean = np.nanmean(health_matrices, axis=0)
    patient_group_mean = np.nanmean(patient_matrices, axis=0)

    # 仅保留 stable_edges_df 中的边，其余置 0
    health_display = np.zeros((N_NODES, N_NODES), dtype=np.float64)
    patient_display = np.zeros((N_NODES, N_NODES), dtype=np.float64)
    for _, row in stable_edges_df.iterrows():
        i, j = int(row["node_i"]), int(row["node_j"])
        health_display[i, j] = health_display[j, i] = health_group_mean[i, j]
        patient_display[i, j] = patient_display[j, i] = patient_group_mean[i, j]

    # Destrieux 148 ROI 坐标（与 绘制群体MSN.ipynb 一致）
    atlas = fetch_atlas_destrieux_2009()
    coords = find_parcellation_cut_coords(atlas["maps"])

    # 健康人群体 MSN
    plt.figure(figsize=(6, 6))
    plotting.plot_connectome(
        health_display,
        coords,
        node_size=10,
        edge_threshold=0,
        display_mode="ortho",
        title="health_group_MSN(Bonferroni)",
    )
    plt.show()

    # 患者群体 MSN
    plt.figure(figsize=(6, 6))
    plotting.plot_connectome(
        patient_display,
        coords,
        node_size=10,
        edge_threshold=0,
        display_mode="ortho",
        title="patient_group_MSN(Bonferroni)",
    )
    plt.show()